# 抽取准备：分步测试

抽取过程中的若干步骤单独拿出来测试，每一步一节。
各节的工具函数之后会提取成脚本。

数据模型见 `docs/designs/graph_model.md`，入库目标是 `infra/neo4j-e08/`：与 04 的样例库（`infra/neo4j/`）数据目录分开，端口相同（bolt 7687），同一时间只起一个。

## 一、种子概念入库

种子在 `data/raw/e08-paper-knowledge/seeds/`，格式与撰写约定见同目录 `README.md`。

### 1.1 加载种子文件

每个种子文件是一条主线的共享节点，顶层只有 `nodes` 与 `relationships`：

```yaml
nodes:
  - ref: res_etth1                  # 文件内引用名；正式 id 入库时分配
    labels: [Resource, Dataset]     # 主 Label + Resource 的次级 Label
    properties:                     # 写进图的属性
      name: ETTh1
      aliases: [...]
      description: >
        ...
      url: ...                      # 仅 Resource
      note: ...                     # 可选
    sources: [...]                  # 核对所用的原始出处，不进图
    verified: true                  # 事实是否已核对，不进图
relationships:
  - {from: res_etth1, type: PART_OF, to: res_ett}   # 无 anchor，出处即种子文件
```

- 主 Label 只有 `Task`、`MethodConcept`、`Resource`、`Metric` 四类。
- 关系只在同一文件的 `ref` 之间：`SUBTYPE_OF`、`OVERLAPS_WITH`、`PART_OF`、`FOR_TASK`。
- 节点和关系都不带 `id`、`anchor`。


In [ ]:
from pathlib import Path

import pandas as pd
import yaml

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'AGENTS.md').is_file())
SEEDS = REPO / "data/raw/e08-paper-knowledge/seeds"
SEED_FILES = sorted(SEEDS.glob("*.yml"))
print([p.name for p in SEED_FILES])

工具函数（之后会提取出来作为脚本使用）

In [ ]:
def load_seed(path: Path) -> dict:
    """读一个种子文件：{"name": 文件名, "nodes": [...], "relationships": [...]}，结构与 yml 相同。"""
    raw = yaml.safe_load(path.read_text(encoding="utf-8"))
    return {"name": path.name, "nodes": raw["nodes"], "relationships": raw["relationships"]}


In [ ]:
seeds = [load_seed(p) for p in SEED_FILES]

nodes_df = pd.DataFrame([
    {"file": s["name"], "ref": n["ref"], "labels": ":".join(n["labels"]), "name": n["properties"]["name"],
     "aliases": len(n["properties"].get("aliases", [])), "note": "note" in n["properties"], "verified": n.get("verified")}
    for s in seeds for n in s["nodes"]
])
rels_df = pd.DataFrame([{"file": s["name"], **r} for s in seeds for r in s["relationships"]])

display(nodes_df.groupby(["labels", "file"]).size().unstack("file", fill_value=0))
display(rels_df.groupby(["type", "file"]).size().unstack("file", fill_value=0))


In [ ]:
nodes_df[nodes_df.verified == False]   # 待核对项


### 1.2 连接 neo4j-e08，建约束与索引

- 13 种主 Label 各建 `id` 唯一约束；
- 三个全文索引：`entity_names`（共享节点的名称与 aliases）、`entity_texts`（共享节点的 description）、`statement_texts`（ClaimConcept、Issue、Claim 的 text 与 description；Claim 只有 text）。
- `note` 不进索引：检索排序回答"是不是查询所指的对象"，靠名称和定义；note 写的是"不是什么、使用时注意什么"，参与排序反而会把写着"不要与 X 合并"的节点在查 X 时提前。note 随候选一起返回给 Agent（第二节）。与 graph_model.md Query 一节（entity_texts 含 note）不一致，待检索层稳定后同步。

分词方式（analyzer）：

- `entity_names` 用默认的 `standard-no-stop-words`：名称短、多是专名和缩写，不做词形还原，不去停用词；前缀、模糊匹配的查询词不经过分词，索引里保留原词形才对得上。
- `entity_texts`、`statement_texts` 用 `english`：去掉 33 个英文停用词，并做词形还原（normalized 与 normalization 归到同一词根），减少 to、and、model 这类泛用词在长文本上累积的噪声。

全部 `IF NOT EXISTS`，重跑无副作用；`CREATE ... IF NOT EXISTS` 不会改已有索引的配置，所以全文索引的覆盖字段或分词方式与声明不一致时，`ensure_schema()` 先删掉旧索引再重建。`library_status()` 打印库里现有的节点与关系，之后每步写入后再调。

In [ ]:
import os

from neo4j import GraphDatabase, RoutingControl

NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_AUTH = (os.environ.get("NEO4J_USER", "neo4j"), os.environ.get("NEO4J_PASSWORD", "password"))
NEO4J_DB = os.environ.get("NEO4J_DATABASE", "neo4j")

# 空库时查不存在的属性键会收到 UNRECOGNIZED 类提示，关掉
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH, notifications_disabled_classifications=["UNRECOGNIZED"])
driver.verify_connectivity()
print(NEO4J_URI, driver.get_server_info().agent)

工具函数（之后会提取出来作为脚本使用）

In [ ]:
LABELS = ["Paper", "Contribution", "Method", "MethodConcept", "Task", "Claim", "ClaimConcept", "Issue",
          "Resource", "ResourceRecord", "Experiment", "Metric", "Observation"]   # graph_model.md 的 13 种 Node
SHARED = "Paper|Method|MethodConcept|Task|Resource|Metric"                      # 带 aliases、按名称查找的共享节点
STATEMENTS = "ClaimConcept|Issue|Claim"                                          # 按陈述文本查找；Claim 只查不复用

# 全文索引：名称 -> (节点, 字段, 分词方式)
FULLTEXT = {
    "entity_names": (SHARED, ["name", "title", "aliases"], "standard-no-stop-words"),
    "entity_texts": (SHARED, ["description"], "english"),
    "statement_texts": (STATEMENTS, ["text", "description"], "english"),
}
SCHEMA = [
    *(f"CREATE CONSTRAINT {l.lower()}_id IF NOT EXISTS FOR (n:{l}) REQUIRE n.id IS UNIQUE" for l in LABELS),
    *(f"CREATE FULLTEXT INDEX {name} IF NOT EXISTS FOR (n:{labels}) ON EACH [{', '.join('n.' + f for f in fields)}] "
      f"OPTIONS {{indexConfig: {{`fulltext.analyzer`: '{analyzer}'}}}}"
      for name, (labels, fields, analyzer) in FULLTEXT.items()),
]

def q(cypher: str, **params) -> list[dict]:
    """只读查询的简写。"""
    records, _, _ = driver.execute_query(cypher, params, database_=NEO4J_DB, routing_=RoutingControl.READ)
    return [r.data() for r in records]

def ensure_schema():
    # 覆盖的 Label、字段或分词方式与声明不一致的全文索引先删掉，下面按声明重建
    existing = {r["name"]: (sorted(r["labels"]), r["properties"], r["analyzer"]) for r in
                q("SHOW FULLTEXT INDEXES YIELD name, labelsOrTypes, properties, options "
                  "RETURN name, labelsOrTypes AS labels, properties, options.indexConfig.`fulltext.analyzer` AS analyzer")}
    for name, (labels, fields, analyzer) in FULLTEXT.items():
        want = (sorted(labels.split("|")), fields, analyzer)
        if name in existing and existing[name] != want:
            print(f"重建 {name}：{existing[name]} -> {want}")
            driver.execute_query(f"DROP INDEX {name}", database_=NEO4J_DB)
    for stmt in SCHEMA:
        driver.execute_query(stmt, database_=NEO4J_DB)
    driver.execute_query("CALL db.awaitIndexes(300)", database_=NEO4J_DB)   # 新建或重建的索引填充完再往下走

def library_status():
    labels = q("MATCH (n) UNWIND labels(n) AS l RETURN l AS label, count(*) AS n ORDER BY label")
    rels = q("MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS n ORDER BY type")
    total = q("MATCH (n) RETURN count(n) AS c")[0]["c"]   # Resource 带两个 label，不能按 label 相加
    print(f"节点 {total}，关系 {sum(r['n'] for r in rels)}")
    if labels:
        display(pd.DataFrame(labels).set_index("label").T)
    if rels:
        display(pd.DataFrame(rels).set_index("type").T)

In [ ]:
ensure_schema()
display(pd.DataFrame(q("SHOW CONSTRAINTS YIELD name, labelsOrTypes, properties")))
display(pd.DataFrame(q("SHOW FULLTEXT INDEXES YIELD name, labelsOrTypes, properties, state, options RETURN name, labelsOrTypes, properties, options.indexConfig.`fulltext.analyzer` AS analyzer, state")))
library_status()

### 1.3 种子写入

逐个种子文件：

1. **查图**：按主 Label + `name` 找节点。已存在就跳过，并取它的 `id`；不存在就交给 id 分配器取下一个。
2. **写入**：一个文件一个写事务，新节点 `CREATE`，只写 `properties`（`sources`、`verified` 不进图）；关系用 `MERGE`，已存在的不会重复建，`OVERLAPS_WITH` 这类语义对称关系不分方向。

**id 分配器**是 notebook 内的全局对象 `ids`：按主 Label 分前缀（如 `task_0001`、`res_0001`），启动时读图中每个前缀已用的最大序号，之后顺序递增。分配后写入失败只会留下空号，不会重号。

跳过的已有节点不会被种子更新；按 `name` 判断存在，节点改名后重跑会被当成新节点。

工具函数（之后会提取出来作为脚本使用）

In [ ]:
PREFIX = {"Paper": "paper", "Contribution": "contrib", "Method": "method", "MethodConcept": "mc",
          "Task": "task", "Claim": "claim", "ClaimConcept": "cc", "Issue": "issue", "Resource": "res",
          "ResourceRecord": "rr", "Experiment": "exp", "Metric": "metric", "Observation": "obs"}
SYMMETRIC = {"DIFFERS_FROM", "OVERLAPS_WITH", "CONTRADICTS"}   # graph_model.md 里的语义对称关系

class IdAllocator:
    """按主 Label 顺序分配 `<prefix>_<4 位序号>`，起点是图中该前缀已用的最大序号。"""

    def __init__(self):
        self.last = {}
        for label, prefix in PREFIX.items():
            rows = q(f"MATCH (n:{label}) WHERE n.id STARTS WITH $p RETURN max(toInteger(substring(n.id, size($p)))) AS m",
                     p=prefix + "_")
            self.last[label] = rows[0]["m"] or 0

    def new(self, label: str) -> str:
        self.last[label] += 1
        return f"{PREFIX[label]}_{self.last[label]:04d}"

def resolve_seed(seed: dict) -> tuple[dict, list]:
    """ref -> id；已在图中的取已有 id，其余新分配。返回 (ref2id, 新节点的 ref 列表)。"""
    ref2id, new_refs = {}, []
    for n in seed["nodes"]:
        main = n["labels"][0]
        hit = q(f"MATCH (n:{main} {{name: $name}}) RETURN n.id AS id", name=n["properties"]["name"])
        if hit:
            ref2id[n["ref"]] = hit[0]["id"]
        else:
            ref2id[n["ref"]] = ids.new(main)
            new_refs.append(n["ref"])
    return ref2id, new_refs

def write_seed(seed: dict, ref2id: dict, new_refs: list) -> int:
    """一个写事务：CREATE 新节点，MERGE 关系。返回新建的关系数。"""
    new = set(new_refs)

    def work(tx):
        for n in seed["nodes"]:
            if n["ref"] in new:
                labels = ":".join(n["labels"])
                tx.run(f"CREATE (n:{labels}) SET n = $props", props={"id": ref2id[n["ref"]], **n["properties"]})
        created = 0
        for r in seed["relationships"]:
            arrow = "-" if r["type"] in SYMMETRIC else "->"
            summary = tx.run(f"MATCH (a {{id: $a}}) MATCH (c {{id: $c}}) MERGE (a)-[:{r['type']}]{arrow}(c)",
                             a=ref2id[r["from"]], c=ref2id[r["to"]]).consume()
            created += summary.counters.relationships_created
        return created

    with driver.session(database=NEO4J_DB) as s:
        return s.execute_write(work)

def ingest_seed(seed: dict) -> dict:
    ref2id, new_refs = resolve_seed(seed)
    rels = write_seed(seed, ref2id, new_refs)
    print(f"{seed['name']}：新建节点 {len(new_refs)}，跳过 {len(ref2id) - len(new_refs)}；"
          f"新建关系 {rels}，已有 {len(seed['relationships']) - rels}")
    return ref2id

In [ ]:
ids = IdAllocator()
print("分配起点", {k: v for k, v in ids.last.items() if v})

ref2id = {}
for seed in seeds:
    ref2id |= ingest_seed(seed)
library_status()

In [ ]:
# 重跑同一批：应全部跳过，id 与第一次一致
for seed in seeds:
    again = ingest_seed(seed)
    assert all(ref2id[r] == i for r, i in again.items())

In [ ]:
# 抽查：ref 与分配到的 id
pd.DataFrame([{"ref": r, "id": i} for r, i in ref2id.items()]).merge(nodes_df[["ref", "name"]]).head(12)

### 1.4 向量补齐与向量索引

为第二节的语义层准备：给需要语义检索的节点算 embedding，存为节点属性 `embedding`，建 Neo4j 向量索引。

- **模型**：Qwen3-Embedding-8B，vLLM 部署在 192.168.163.112:8002（`infra/vllm-qwen3-embed/`），4096 维，返回已归一化，相似度用 cosine。
- **写入侧不加任务说明**，只有查询侧加（第二节）。
- **向量的输入文本**：共享节点用 `name`、`aliases` 和 `description` 拼成一段；Issue 用 `text` 和 `description`（问题文本短，范围和条件写在 description 里）；ClaimConcept、Claim 只有 `text`。`note` 不进向量，理由同 1.2。
- **两个向量索引**：`entity_vectors`（共享节点）与 `statement_vectors`（ClaimConcept、Issue、Claim）。
- **补齐而非重算**：每个节点另存 `embedding_key`（模型名 + 输入文本的哈希）。`sync_embeddings()` 只给没有向量或 key 对不上（文本改过、换了模型）的节点计算，所以重跑无副作用，之后每次写入图谱后调一次即可。

`embedding`、`embedding_key` 是检索用的派生属性，不属于 graph_model 的内容属性。

工具函数（之后会提取出来作为脚本使用）

In [ ]:
import hashlib

import httpx

EMBED_URL = os.environ.get("EMBED_URL", "http://192.168.163.112:8002/v1/embeddings")
EMBED_MODEL = os.environ.get("EMBED_MODEL", "qwen3-embedding-8b")
EMBED_DIM = 4096
EMBED_BATCH = 32

VECTOR_SCHEMA = [
    f"CREATE VECTOR INDEX {name} IF NOT EXISTS FOR (n:{labels}) ON n.embedding "
    f"OPTIONS {{indexConfig: {{`vector.dimensions`: {EMBED_DIM}, `vector.similarity_function`: 'cosine'}}}}"
    for name, labels in [("entity_vectors", SHARED), ("statement_vectors", STATEMENTS)]
]

def embed(texts: list[str]) -> list[list[float]]:
    out = []
    for i in range(0, len(texts), EMBED_BATCH):
        r = httpx.post(EMBED_URL, json={"model": EMBED_MODEL, "input": texts[i:i + EMBED_BATCH]}, timeout=120)
        r.raise_for_status()
        out += [d["embedding"] for d in sorted(r.json()["data"], key=lambda d: d["index"])]
    assert all(len(v) == EMBED_DIM for v in out)
    return out

def embedding_text(labels: list[str], props: dict) -> str | None:
    """节点用于向量的输入文本；没有可用文本时返回 None。"""
    if labels[0] in STATEMENTS.split("|"):
        parts = [props.get("text"), props.get("description") if labels[0] == "Issue" else None]
        return "\n".join(p.strip() for p in parts if p) or None
    name = props.get("name") or props.get("title")
    if not name:
        return None
    parts = [name]
    if props.get("aliases"):
        parts.append("Also known as: " + ", ".join(props["aliases"]))
    if props.get("description"):
        parts.append(props["description"].strip())
    return "\n".join(parts)

def embedding_key(text: str) -> str:
    return hashlib.sha1(f"{EMBED_MODEL}\n{text}".encode()).hexdigest()[:16]

def sync_embeddings() -> int:
    """给缺向量或向量过期的节点补算；返回补算的节点数。"""
    rows = q(f"""MATCH (n) WHERE n:{SHARED} OR n:{STATEMENTS}
                 RETURN n.id AS id, labels(n) AS labels, n.embedding_key AS key,
                        n {{.name, .title, .aliases, .description, .text}} AS props""")
    todo = []
    for r in rows:
        text = embedding_text(r["labels"], r["props"])
        if text and r["key"] != embedding_key(text):
            todo.append({"id": r["id"], "text": text, "key": embedding_key(text)})
    if todo:
        for row, v in zip(todo, embed([t["text"] for t in todo])):
            row["v"] = v
        driver.execute_query("""UNWIND $rows AS row MATCH (n {id: row.id})
                                CALL db.create.setNodeVectorProperty(n, 'embedding', row.v)
                                SET n.embedding_key = row.key""",
                             rows=[{k: t[k] for k in ("id", "key", "v")} for t in todo], database_=NEO4J_DB)
    return len(todo)

In [ ]:
for stmt in VECTOR_SCHEMA:
    driver.execute_query(stmt, database_=NEO4J_DB)

print("补算向量", sync_embeddings(), "个节点；重跑补算", sync_embeddings(), "个")
display(pd.DataFrame(q("SHOW VECTOR INDEXES YIELD name, labelsOrTypes, properties, state")))
display(pd.DataFrame(q(f"""MATCH (n:{SHARED}) RETURN n.embedding IS NOT NULL AS has_embedding, count(*) AS n""")))

In [ ]:
# 链路检查：带任务说明的查询向量，直接查 entity_vectors
INSTRUCT = ("Instruct: Given a mention of a dataset, method, task or metric in a research paper, "
            "retrieve the entry that refers to the same object\nQuery:")
v = embed([INSTRUCT + "RevIN: reversible instance normalization applied to model inputs and outputs"])[0]
# Neo4j 2026 用 Cypher 25 的 SEARCH 子句（db.index.vector.queryNodes 已弃用）；cosine 分数已映射到 [0, 1]
pd.DataFrame(q("""CYPHER 25
                  MATCH (node) SEARCH node IN (VECTOR INDEX entity_vectors FOR $v LIMIT 5) SCORE AS score
                  RETURN node.id AS id, node.name AS name, round(score, 3) AS score""", v=v))

## 二、入库前的查询算子 `find_entities`

Agent 读到一个称呼（mention），写入库数据前先查图：复用已有节点、拿它的 `id` 连边，还是新建。

算子对外只有一个入口，内部怎么组合索引对 Agent 不可见：

```python
find_entities(mention, description=None, entity_type=None, n=5)
```

- `mention`：原文里的称呼。
- `description`：可选，Agent 读原文后写的一句说明，与节点的 `description` 是同一种文字（如 RevIN → `reversible instance normalization applied to model inputs and outputs`）。只给名称时，只能按名称和整体语义找。
- `entity_type`：可选，主 Label 或次级 Label（如 `Method`、`Dataset`）。

算子**不替 Agent 做判断**：返回排好序的候选与证据，不给置信度、不设阈值。查询质量的系统测试等六篇入库后，用抽取运行中的真实查询与 Agent 的决定再做。

### 2.1 两个阶段

**Stage 1：标识解析（identifier）。** mention 与节点的 `name` / `title` / `aliases` 比较，命中即为强证据：

- `name` / `alias`：去首尾空白、转小写后相等；
- `name (normalized)` / `alias (normalized)`：只保留字母和数字后相等（`time-series forecasting` 与 `time series forecasting`、`Llama 3 8B` 与 `Llama-3-8B`）。

不做缩写还原：缩写对应什么只能查对照表，而这张表就是 aliases。精度高、召回低，所以不参与下面的融合排序，命中的候选固定排在最前。alias 不是唯一键，可能不止一个候选命中。

**Stage 2：混合召回与融合。** 无论 Stage 1 是否命中都执行：命中后 Agent 仍需要看到相近的节点（查 ETTh1 时的 ETT、同系列的其他型号）。三个通道各召回 `POOL`（20）个，各比各的：

| 通道 | 查询侧 | 节点侧 | 方式 |
|---|---|---|---|
| `lexical_name` | mention 的每个词：原词、前缀（`etth1*`）、模糊（`etth1~`） | `name`、`title`、`aliases`（`entity_names`，默认分词） | BM25 |
| `lexical_text` | description 的词；没给 description 时不执行 | `description`（`entity_texts`，`english` 分词） | BM25 |
| `semantic` | 任务说明 + `mention: description` 的向量 | `name`、`aliases`、`description` 拼成的向量（`entity_vectors`） | cosine |

三个通道按 RRF 融合，只用名次，不比较量纲不同的原始分数：

$$
\text{rrf}(e) = \sum_{c \in \text{Stage 2}} \frac{1}{k + r_c(e)}
$$

$r_c(e)$ 是候选 $e$ 在通道 $c$ 中的名次，未被召回的通道不计。$k$ 暂取 10：每路只有 20 条，常用的 60 会让名次几乎不起作用，排序只剩"被几个通道召回"。

`note` 不参与任何通道，但随候选返回：区分用的提醒（如"与 Llama-3-70B-Instruct 是不同模型"）要在 Agent 看候选时出现。

**输出**：Stage 1 命中的在前，其后是 Stage 2 按 RRF 排序的候选，截取前 `n` 个。每个候选带 id、Label、名称、aliases、description、note、一跳邻接关系，以及 `evidence`（召回它的每个通道：Stage 1 的命中方式，Stage 2 的名次与原始分数）。

已知的偏差：`lexical_text` 与 `semantic` 都读到节点的 description，同一候选被两路召回，部分是重复计数。先记下，有数据后再看影响。

工具函数（之后会提取出来作为脚本使用）

In [ ]:
import re

POOL = 20      # 每个通道召回的条数
RRF_K = 10
TOP_N = 5
ENTITY_INSTRUCT = ("Instruct: Given a mention of a dataset, method, task or metric in a research paper, "
                   "retrieve the entry that refers to the same object\nQuery:")
TYPE_FILTER = "($type IS NULL OR $type IN labels(node))"

def _tokens(s: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", s.lower())

def _normalize(s: str) -> str:
    """Stage 1 的宽松比较：只保留字母和数字的小写串。与 Cypher 里的 string.regexReplace 同一规则。"""
    return re.sub(r"[^a-z0-9]", "", s.lower())

def _identifier(mention, entity_type, pool):
    rows = q(f"""CYPHER 25
                 WITH toLower(trim($m)) AS m, $mn AS mn
                 MATCH (node:{SHARED}) WHERE {TYPE_FILTER}
                 WITH node, m, mn, coalesce(node.name, node.title) AS nm, coalesce(node.aliases, []) AS al
                 WITH node, CASE
                     WHEN toLower(trim(nm)) = m THEN 'name'
                     WHEN ANY(a IN al WHERE toLower(trim(a)) = m) THEN 'alias'
                     WHEN mn <> '' AND string.regexReplace(toLower(nm), '[^a-z0-9]', '') = mn THEN 'name (normalized)'
                     WHEN mn <> '' AND ANY(a IN al WHERE string.regexReplace(toLower(a), '[^a-z0-9]', '') = mn)
                         THEN 'alias (normalized)'
                 END AS match
                 WHERE match IS NOT NULL
                 RETURN node.id AS id, match ORDER BY match, id LIMIT $pool""",
             m=mention, mn=_normalize(mention), type=entity_type, pool=pool)
    return [{"id": r["id"], "match": r["match"]} for r in rows]

def _fulltext(index, query, entity_type, pool):
    if not query:
        return []
    return q(f"""CALL db.index.fulltext.queryNodes('{index}', $query) YIELD node, score
                 WHERE {TYPE_FILTER}
                 RETURN node.id AS id, score ORDER BY score DESC LIMIT $pool""",
             query=query, type=entity_type, pool=pool)

def _semantic(text, entity_type, pool):
    v = embed([ENTITY_INSTRUCT + text])[0]
    # 类型过滤在索引之后做，所以先多取一些
    return q(f"""CYPHER 25
                 MATCH (node) SEARCH node IN (VECTOR INDEX entity_vectors FOR $v LIMIT $k) SCORE AS score
                 WITH node, score WHERE {TYPE_FILTER}
                 RETURN node.id AS id, score ORDER BY score DESC LIMIT $pool""",
             v=v, k=pool * 5, type=entity_type, pool=pool)

def recall_channels(mention, description=None, entity_type=None, pool=POOL) -> dict[str, list[dict]]:
    """召回：Stage 1 的 identifier 与 Stage 2 的三个通道，各自的有序结果 [{id, score | match}, ...]。"""
    return {
        "identifier": _identifier(mention, entity_type, pool),
        "lexical_name": _fulltext("entity_names", " ".join(f"{t} {t}* {t}~" for t in _tokens(mention)), entity_type, pool),
        "lexical_text": _fulltext("entity_texts", " ".join(_tokens(description or "")), entity_type, pool),
        "semantic": _semantic(f"{mention}: {description}" if description else mention, entity_type, pool),
    }

def fuse(channels: dict[str, list[dict]], k=RRF_K) -> list[tuple[str, dict, float]]:
    """融合：Stage 1 命中的在前，其余按 Stage 2 的 RRF。返回 [(id, evidence, rrf)]，evidence = {通道: {rank, score | match}}。"""
    evidence = {}
    for ch, rows in channels.items():
        for rank, r in enumerate(rows, 1):
            evidence.setdefault(r["id"], {})[ch] = {"rank": rank, **{k_: r[k_] for k_ in ("score", "match") if k_ in r}}
    # RRF 只在 Stage 2 的通道之间计算；identifier 命中的另行置顶
    rrf = {i: sum(1 / (k + e["rank"]) for ch, e in ev.items() if ch != "identifier") for i, ev in evidence.items()}
    pinned = [r["id"] for r in channels["identifier"]]
    rest = sorted((i for i in evidence if i not in pinned), key=lambda i: (-rrf[i], i))
    return [(i, evidence[i], rrf[i]) for i in pinned + rest]

def _entity_info(ids: list[str]) -> dict:
    """候选的定义与一跳邻接关系（箭头表示方向）。"""
    rows = q("""MATCH (node) WHERE node.id IN $ids
                OPTIONAL MATCH (node)-[r]-(other)
                RETURN node.id AS id, labels(node) AS labels, coalesce(node.name, node.title) AS name,
                       node.aliases AS aliases, node.description AS description, node.note AS note,
                       [x IN collect(CASE WHEN r IS NULL THEN null
                                          WHEN startNode(r) = node THEN type(r) + ' → ' + coalesce(other.name, other.title, other.id)
                                          ELSE type(r) + ' ← ' + coalesce(other.name, other.title, other.id) END)
                        WHERE x IS NOT NULL] AS neighbors""", ids=ids)
    return {r["id"]: r for r in rows}

def find_entities(mention: str, description: str | None = None, entity_type: str | None = None,
                  n: int = TOP_N) -> list[dict]:
    """查询算子：召回 → 融合 → 截断。返回前 n 个候选，每个带实体信息与 evidence；不给置信度。"""
    top = fuse(recall_channels(mention, description, entity_type))[:n]
    info = _entity_info([i for i, _, _ in top])
    return [{**info[i], "evidence": ev} for i, ev, _ in top]

浏览用的两个视图：`show_candidates` 是算子的实际输出；`show_channels` 是调试视图，列出融合前各通道的前几名，用来对照融合丢掉或提上来了什么。

In [ ]:
def _fmt_evidence(ev: dict) -> str:
    return "; ".join(f"{ch} #{e['rank']}" + (f" {e['match']}" if "match" in e else f" {e['score']:.3f}")
                     for ch, e in ev.items())

def show_candidates(mention, description=None, entity_type=None, n=TOP_N):
    rows = find_entities(mention, description, entity_type, n)
    print(f"{mention!r}" + (f"  description={description!r}" if description else "")
          + (f"  type={entity_type}" if entity_type else ""))
    df = pd.DataFrame([{
        "id": r["id"], "labels": ":".join(r["labels"]), "name": r["name"],
        "evidence": _fmt_evidence(r["evidence"]), "description": (r["description"] or "")[:70],
        "neighbors": "; ".join(r["neighbors"]),
    } for r in rows])
    display(df.style.set_properties(subset=["evidence", "description", "neighbors"], **{"white-space": "pre-wrap"}))

def show_channels(mention, description=None, entity_type=None, head=TOP_N):
    """调试视图：各通道前 head 名，以及融合后的名次与 RRF 值。"""
    channels = recall_channels(mention, description, entity_type)
    fused = {i: (pos, rrf) for pos, (i, _, rrf) in enumerate(fuse(channels), 1)}
    ids = list(dict.fromkeys(r["id"] for rows in channels.values() for r in rows[:head]))
    names = {i: r["name"] for i, r in _entity_info(ids).items()}
    df = pd.DataFrame([{
        "fused": fused[i][0], "rrf": round(fused[i][1], 3), "name": names[i],
        **{ch: next((f"#{p}" for p, r in enumerate(rows, 1) if r["id"] == i), "") for ch, rows in channels.items()},
    } for i in ids]).sort_values("fused")
    display(df.set_index("fused"))

In [ ]:
show_candidates("ETTh1")                    # identifier 命中；ETT 与其他子集是否跟着出现

In [ ]:
show_candidates("LTSF")                     # alias 命中

In [ ]:
show_candidates("Llama 3 8B Instruct")   # 只差连字符与空格：Stage 1 的 normalized 命中

In [ ]:
show_candidates("RevIN")                                                                          # 只给名称
show_candidates("RevIN", "reversible instance normalization applied to model inputs and outputs")    # 加 description

In [ ]:
show_candidates("gpt-3.5-turbo-1106", entity_type="Model")   # 快照比节点细：模型族节点能否排前

In [ ]:
show_candidates("PatchTST", "patch-based channel-independent Transformer for long-term forecasting")   # 图里没有

In [ ]:
# 调试：融合前后对照
show_channels("RevIN", "reversible instance normalization applied to model inputs and outputs")

### 2.2 `find_statements`：命题与问题

```python
find_statements(text, description=None, kind=None, n=5)
```

- `text`：Agent 从原文整理出的一句命题或一个问题。
- `description`：可选，只对问题有意义：问题的范围、条件。
- `kind`：可选，`ClaimConcept`、`Issue` 或 `Claim`。

查找对象是 ClaimConcept、Issue，以及各论文的 Claim。Claim 属于论文局部节点，**只查不复用**：第二篇论文说出与第一篇相同的命题、而图中还没有对应的 ClaimConcept 时，只有查到第一篇的 Claim，Agent 才知道该新建 ClaimConcept 并把两条 Claim 都连上去。返回的 Claim 标 `reusable: false`，并带所属论文。

与 `find_entities` 的差别：命题没有称呼，没有 Stage 1，也没有名称通道，只有两路召回：

| 通道 | 查询侧 | 节点侧 | 方式 |
|---|---|---|---|
| `lexical_text` | text 与 description 的词 | `text`、`description`（`statement_texts`，`english` 分词） | BM25 |
| `semantic` | 任务说明 + text（+ description）的向量 | ClaimConcept、Claim 的 `text`；Issue 的 `text` + `description`（`statement_vectors`） | cosine |

两路各召回 `POOL` 个，按 RRF（k = `RRF_K`）融合，截取前 `n` 个。每个候选带 id、Label、text、description、note、`reusable`、所属论文（仅 Claim）、一跳邻接关系与 `evidence`。

措辞相近不等于同一命题：条件、对象或口径不同的两条主张不应合并，这由 Agent 对照原文判断。

工具函数（之后会提取出来作为脚本使用）

In [ ]:
STATEMENT_INSTRUCT = ("Instruct: Given a claim or research question from a paper, "
                      "retrieve statements that express the same proposition or ask the same question\nQuery:")
KIND_FILTER = "($kind IS NULL OR $kind IN labels(node))"

def recall_statement_channels(text, description=None, kind=None, pool=POOL) -> dict[str, list[dict]]:
    query_text = f"{text}\n{description}" if description else text
    lexical = q(f"""CALL db.index.fulltext.queryNodes('statement_texts', $query) YIELD node, score
                    WHERE {KIND_FILTER}
                    RETURN node.id AS id, score ORDER BY score DESC LIMIT $pool""",
                query=" ".join(_tokens(query_text)), kind=kind, pool=pool) if _tokens(query_text) else []
    v = embed([STATEMENT_INSTRUCT + query_text])[0]
    semantic = q(f"""CYPHER 25
                     MATCH (node) SEARCH node IN (VECTOR INDEX statement_vectors FOR $v LIMIT $k) SCORE AS score
                     WITH node, score WHERE {KIND_FILTER}
                     RETURN node.id AS id, score ORDER BY score DESC LIMIT $pool""",
                 v=v, k=pool * 5, kind=kind, pool=pool)
    return {"lexical_text": lexical, "semantic": semantic}

def _statement_info(ids: list[str]) -> dict:
    """候选的陈述、所属论文（Claim）与一跳邻接关系；对端没有名称时显示截短的 text。"""
    rows = q("""MATCH (node) WHERE node.id IN $ids
                OPTIONAL MATCH (paper:Paper)-[:HAS_CLAIM]->(node)
                OPTIONAL MATCH (node)-[r]-(other)
                WITH node, collect(DISTINCT paper.id) AS papers, r, other,
                     coalesce(other.name, other.title, left(other.text, 60), other.id) AS other_name
                RETURN node.id AS id, labels(node) AS labels, node.text AS text,
                       node.description AS description, node.note AS note, papers,
                       [x IN collect(CASE WHEN r IS NULL THEN null
                                          WHEN startNode(r) = node THEN type(r) + ' → ' + other_name
                                          ELSE type(r) + ' ← ' + other_name END)
                        WHERE x IS NOT NULL] AS neighbors""", ids=ids)
    return {r["id"]: {**r, "reusable": r["labels"][0] != "Claim"} for r in rows}

def find_statements(text: str, description: str | None = None, kind: str | None = None,
                    n: int = TOP_N) -> list[dict]:
    """查询算子：两路召回 → RRF 融合 → 截断。Claim 标 reusable=False 并带所属论文；不给置信度。"""
    top = fuse({"identifier": [], **recall_statement_channels(text, description, kind)})[:n]
    info = _statement_info([i for i, _, _ in top])
    return [{**info[i], "evidence": ev} for i, ev, _ in top]

def show_statements(text, description=None, kind=None, n=TOP_N):
    rows = find_statements(text, description, kind, n)
    print(f"{text!r}" + (f"  kind={kind}" if kind else "") + f"：{len(rows)} 个候选")
    if rows:
        display(pd.DataFrame([{
            "id": r["id"], "labels": ":".join(r["labels"]), "reusable": r["reusable"], "papers": ", ".join(r["papers"]),
            "evidence": _fmt_evidence(r["evidence"]), "text": (r["text"] or "")[:90],
            "neighbors": "; ".join(r["neighbors"]),
        } for r in rows]))

In [ ]:
# 目前图中还没有 ClaimConcept、Issue、Claim：这里只确认链路能跑通、返回为空；效果等首篇抽取后再看
show_statements("A simple linear model outperforms Transformer-based models on long-term time series forecasting benchmarks")
show_statements("Are Transformers effective for long-term time series forecasting?", kind="Issue")